In [1]:
import sys
import os
import duckdb

# 
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.append(project_root)

from backend.configs.database import DB_PATH
from transform_scripts.classifier import get_classification_scores, FETCH_UNCATEGORIZED_CASES

/home/david/git-repos/danish-politician-votes/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2026-04-19 20:35:15,031 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/MoritzLaurer/mDeBERTa-v3-base-mnli-xnli/resolve/main/config.json "HTTP/1.1 307 Temporary Redirect"
2026-04-19 20:35:15,079 - httpx - INFO - HTTP Request: HEAD https://huggingface.co/api/resolve-cache/models/MoritzLaurer/mDeBERTa-v3-base-mnli-xnli/8adb042d524ecd5c26d3e3ba0e3fbcf7e2d0864c/config.json "HTTP/1.1 200 OK"
Loading weights: 100%|███████████████████████████████████████████████████████████████████████████████████████████████████████████████| 202/202 [00:00<00:00, 5290.18it/s]
DebertaV2ForSequenceClassification LOAD REPORT from: MoritzLaurer/mDeBERTa-v3-base-mnli-xnli
Key                             | Status     |  | 

In [2]:
classifier_version = 2
cat_version = 4

In [3]:
fetch_qry = FETCH_UNCATEGORIZED_CASES.format(
    cat_version=cat_version,
    classifier_version=classifier_version,
    size=32
)

In [4]:
conn = duckdb.connect(database=DB_PATH)
df = conn.execute(fetch_qry).fetchdf()

df['categorization_text'] = df['case_title'].fillna('') + ': ' + df['case_reasoning'].fillna('')
texts = df['categorization_text'].tolist()

In [5]:
batch_scores = get_classification_scores(texts)

In [6]:
batch_scores

,sequence,"Kultur, Turisme og Fritid:Dette handler om kultur, turisme, sport, medier, fritidsaktiviteter, kunst, museer eller events (ikke økonomiske aspekter af turisme).","Integration og Migration:Dette handler om integration, immigration, asyl, flygtninge, statsborgerskab eller udlændingepolitik (ikke sociale ydelser til indvandrere).","Miljø, Klima og Energi:Dette handler om miljø, klima, bæredygtighed, energipolitik, affald, naturfredning eller grøn omstilling (ikke byplanlægning, transport eller landbrug).","Udenrigs- og EU-anliggender:Dette handler **udelukkende** om internationale forhandlinger, EU-samarbejde, udenrigspolitik eller traktater (ikke indhold om emner som klima, handel eller sikkerhed, medmindre teksten udelukkende omhandler den internationale proces).","Uddannelse og Forskning:Dette handler om uddannelse, skoler, universiteter, forskning, innovation, digitalisering i undervisning eller videnskabelige bevillinger (ikke ren teknologiudvikling eller erhvervsrettede uddannelser).","Infrastruktur og Transport:Dette handler om fysisk infrastruktur, transport, boliger, byplanlægning, veje, kollektiv transport eller byggeprojekter (ikke digital infrastruktur eller energinet).","Økonomi og Beskæftigelse:Dette handler om økonomi, beskæftigelse, skatter, budgetter, erhvervslovgivning, arbejdsmarked, virksomhedsstøtte eller finansielle bevillinger (ikke miljø, social velfærd eller forskning).","Social og Sundhed:Dette handler om social velfærd, sundhedsvæsen, ældrepleje, handicap, folkesundhed, sygehuse eller sociale ydelser (ikke økonomiske bevillinger til sundhedsvæsenet).","Landbrug, Fødevarer og Fiskeri:Dette handler om landbrug, fødevarer, fiskeri, dyrevelfærd eller landdistrikters udvikling (ikke miljøregulering eller international handel).","Sikkerhed og Retsvæsen:Dette handler om sikkerhed, kriminalitet, politi, forsvar, retssystemet, cyberkriminalitet eller lovgivning om straffe (ikke international sikkerhedspolitik)."
0,"Forslag til folketingsbeslutning om, at Danmar...",0.761804,0.561215,0.497559,0.447702,0.106970,0.012871,0.006552,0.000376,0.000274,0.000187
1,Forslag til folketingsbeslutning om at sikre å...,0.001084,0.734596,0.768038,0.309858,0.369770,0.093574,0.190423,0.000327,0.000364,0.000298
2,Forslag til lov om ændring af lov om Institut ...,0.511839,0.131062,0.420243,0.011497,0.001731,0.002726,0.006539,0.000647,0.000277,0.000207
3,Forslag til lov om tillægsbevilling for finans...,0.001727,0.013956,0.630457,0.469764,0.262558,0.006438,0.017029,0.000208,0.000159,0.000170
4,Forslag til lov om ændring af dyrevelfærdslove...,0.227856,0.016231,0.178544,0.036718,0.002095,0.000942,0.006771,0.000394,0.000185,0.000352
5,Forslag til lov om indfødsrets meddelelse.:,0.935894,0.133698,0.584034,0.348645,0.084720,0.010449,0.006205,0.000364,0.000183,0.000188
6,Forslag til lov om ændring af lov om leje. (Di...,0.780328,0.594522,0.648240,0.153422,0.030129,0.000121,0.036494,0.000958,0.000202,0.000513
7,"Forslag til lov om ændring af SU-loven, lov om...",0.222193,0.010530,0.338296,0.129899,0.037257,0.003248,0.001875,0.000377,0.000172,0.000222
8,Forslag til lov om ændring af udlændingeloven ...,0.731029,0.216094,0.737205,0.210523,0.002627,0.005261,0.009313,0.000437,0.000187,0.000206
9,Forslag til lov om ændring af lov om social se...,0.891528,0.002155,0.699737,0.332636,0.037715,0.028707,0.003279,0.000863,0.000217,0.000289


In [7]:
texts

['Forslag til folketingsbeslutning om, at Danmark skal anerkende Palæstina.: ',
 'Forslag til folketingsbeslutning om at sikre åbenhed om udgifter afholdt af et andet ministerium end ministerens eget.\n: ',
 'Forslag til lov om ændring af lov om Institut for Menneskerettigheder – Danmarks Nationale Menneskerettighedsinstitution. (Tilføjelse af religion eller tro, alder og handicap til Institut for Menneskerettigheders kompetenceområde).: ',
 'Forslag til lov om tillægsbevilling for finansåret 2025.: ',
 'Forslag til lov om ændring af dyrevelfærdsloven, straffeloven og retsplejeloven. (Strafskærpelser og udvidet mulighed for rettighedsfrakendelse på dyrevelfærdsområdet).: ',
 'Forslag til lov om indfødsrets meddelelse.: ',
 'Forslag til lov om ændring af lov om leje. (Digital kommunikation om særlig bebyrdende meddelelser).: ',
 'Forslag til lov om ændring af SU-loven, lov om Udbetaling Danmark, lov om Arbejdsgivernes Uddannelsesbidrag og forskellige andre love. (Overførsel af opgaver f